In [13]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
import pickle

with open("vehicle_split.pkl","rb") as f:
    data =pickle.load(f)

print(type(data))

<class 'dict'>


In [3]:
print(data.keys())

dict_keys(['X_train', 'X_val', 'y_train', 'y_val'])


In [4]:
X_train = data["X_train"]
X_val = data["X_val"]
y_train = data["y_train"]
y_val = data["y_val"]
print("Train samples:", len(X_train))
print("Validation samples:", len(X_val))

print("First image path:",X_train[0])
print("First label:",y_train[0])

Train samples: 1537
Validation samples: 385
First image path: C:\Users\ASUS\Desktop\مکتب شریف\مکتب 145\پروژه دوم\data\train - merge - clean\kamyunet\215916570.jpg
First label: 3


In [5]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

print("Project path:", Path.cwd().parent)

Project path: c:\Users\ASUS\Desktop\مکتب شریف\مکتب 145\پروژه دوم


In [6]:
from src.data_loader import create_dataloaders

In [7]:
train_loader ,val_loader = create_dataloaders(
    X_train,
    X_val,
    y_train,
    y_val,
    batch_size=32
)

In [8]:
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

Train batches: 49
Validation batches: 13
Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Image dtype: torch.float32
Label dtype: torch.int64


In [11]:
# CNN baseline 

class SimpleCNN(nn.Module):
    def __init__(self,num_classes=8):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*56*56,128),
            nn.ReLU(),
            nn.Linear(128,num_classes)
        )


    def forward(self,x) : 
        x = self.features(x)
        x = self.classifier(x)
        return x

In [12]:
model = SimpleCNN(num_classes = 8).to(device)
print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=200704, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=8, bias=True)
  )
)


In [13]:
images , labels = next(iter(train_loader))

images = images.to(device)

outputs = model(images)

print("images : ", images.shape)
print("outputs", outputs.shape)

images :  torch.Size([32, 3, 224, 224])
outputs torch.Size([32, 8])


In [14]:
criterion = nn.CrossEntropyLoss()
print(criterion)

CrossEntropyLoss()


In [15]:
labels = labels.to(device)
loss = criterion(outputs,labels)
print("Loss :",loss.item() )

Loss : 2.0753426551818848


In [17]:
optimizer = optim.Adam(
    model.parameters(),
    lr = 0.001
)
print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [18]:
print("Outputs shape:", outputs.shape)
print("Labels shape:", labels.shape)
print("Labels dtype:", labels.dtype)

Outputs shape: torch.Size([32, 8])
Labels shape: torch.Size([32])
Labels dtype: torch.int64


In [19]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Images dtype:", images.dtype)

print("Labels shape:", labels.shape)
print("Labels dtype:", labels.dtype)

Images shape: torch.Size([32, 3, 224, 224])
Images dtype: torch.float32
Labels shape: torch.Size([32])
Labels dtype: torch.int64


In [20]:
images = images.to(device)
labels = labels.to(device)

outputs = model(images)

loss = criterion(outputs, labels)

optimizer.zero_grad()

loss.backward()

optimizer.step()

print("Training loss:", loss.item())

Training loss: 2.079660415649414


In [21]:
images,labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

optimizer.zero_grad()

outputs = model(images)

loss = criterion(outputs,labels)

loss.backward()

optimizer.step()

print("training loss : ",loss.item())

training loss :  9.892814636230469


In [22]:
num_epochs = 5
for epoch in range(num_epochs):
    model.train()

    running_loss = 0

    for images,labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss/len(train_loader)

    print(
        f"Epoch : [{epoch+1}/{num_epochs}]"
        f"Train loss : {epoch_loss}"
    )

KeyboardInterrupt: 

In [23]:
model.eval()
val_loss = 0

with torch.no_grad():
    for images,labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
    
            optimizer.zero_grad()
    
            outputs = model(images)
    
            loss = criterion(outputs,labels)
    
            val_loss += loss.item()
    
val_loss = val_loss/len(val_loader)
    
print("validation loss:",val_loss
)

validation loss: 2.0009821378267727


In [24]:
model = SimpleCNN(num_classes = 8).to(device)
criterion = nn.CrossEntropyLoss()


optimizer = optim.Adam(
model.parameters(),
lr = 0.001

)

print("Model and optimizer are ready.")

Model and optimizer are ready.


In [22]:
num_epochs = 10

for epoch in range(num_epochs):

    # =====================
    # Training
    # =====================
    model.train()

    running_train_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

    train_loss = running_train_loss / len(train_loader)


    # =====================
    # Validation
    # =====================
    model.eval()

    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = 100 * correct / total


    # =====================
    # Results
    # =====================
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}%"
    )

Epoch [1/10] Train Loss: 2.2787343555567214 | Val Loss: 1.5666 | Val Accuracy: 43.90%
Epoch [2/10] Train Loss: 1.2765004087467582 | Val Loss: 1.1530 | Val Accuracy: 59.48%
Epoch [3/10] Train Loss: 0.9398454756152873 | Val Loss: 0.9533 | Val Accuracy: 65.45%
Epoch [4/10] Train Loss: 0.655075059861553 | Val Loss: 0.8765 | Val Accuracy: 68.57%
Epoch [5/10] Train Loss: 0.4420052905167852 | Val Loss: 0.7784 | Val Accuracy: 72.73%
Epoch [6/10] Train Loss: 0.25745199158151005 | Val Loss: 0.7736 | Val Accuracy: 72.99%
Epoch [7/10] Train Loss: 0.1416654421054587 | Val Loss: 0.8656 | Val Accuracy: 73.51%
Epoch [8/10] Train Loss: 0.08087479493138855 | Val Loss: 0.8291 | Val Accuracy: 76.10%
Epoch [9/10] Train Loss: 0.04804365406269018 | Val Loss: 0.8757 | Val Accuracy: 75.06%
Epoch [10/10] Train Loss: 0.013728362337058904 | Val Loss: 0.9547 | Val Accuracy: 76.36%


In [ ]:
from torchvision import models

resnet18 = models.resnet18(weights= "DEFAULT")

print(resnet18)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [26]:
resnet18.fc = nn.Linear(
    resnet18.fc.in_features,
    8
)

print(resnet18.fc)

Linear(in_features=512, out_features=8, bias=True)


In [28]:
for param in resnet18.parameters():
    param.requires_grad = False

for param in resnet18.fc.parameters():
    param.requires_grad = True


print("Backbone frozen.")
print("Final layer trainable.")


Backbone frozen.
Final layer trainable.


In [29]:
resnet18 = resnet18.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    resnet18.fc.parameters(),
    lr = 0.001
)
print("ResNet18 is ready for training.")

ResNet18 is ready for training.


In [30]:
images,labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = resnet18(images)

print("Images shape:", images.shape)
print("Outputs shape:", outputs.shape)
print("Labels shape:", labels.shape)

Images shape: torch.Size([32, 3, 224, 224])
Outputs shape: torch.Size([32, 8])
Labels shape: torch.Size([32])


In [28]:
num_epochs = 10

for epoch in range(num_epochs):

    # =====================
    # Training
    # =====================
    resnet18.train()

    running_train_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = resnet18(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

    train_loss = running_train_loss / len(train_loader)


    # =====================
    # Validation
    # =====================
    resnet18.eval()

    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = resnet18(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = 100 * correct / total


    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}%"
    )

Epoch [1/10] Train Loss: 1.6720 | Val Loss: 1.2158 | Val Accuracy: 69.35%
Epoch [2/10] Train Loss: 1.0323 | Val Loss: 0.8518 | Val Accuracy: 82.60%
Epoch [3/10] Train Loss: 0.7928 | Val Loss: 0.6869 | Val Accuracy: 84.16%
Epoch [4/10] Train Loss: 0.6762 | Val Loss: 0.5644 | Val Accuracy: 86.49%
Epoch [5/10] Train Loss: 0.6042 | Val Loss: 0.5197 | Val Accuracy: 86.23%
Epoch [6/10] Train Loss: 0.5546 | Val Loss: 0.4794 | Val Accuracy: 88.31%
Epoch [7/10] Train Loss: 0.5037 | Val Loss: 0.4466 | Val Accuracy: 88.31%
Epoch [8/10] Train Loss: 0.4591 | Val Loss: 0.4207 | Val Accuracy: 89.35%
Epoch [9/10] Train Loss: 0.4128 | Val Loss: 0.4041 | Val Accuracy: 88.31%
Epoch [10/10] Train Loss: 0.4344 | Val Loss: 0.4089 | Val Accuracy: 89.35%


In [ ]:
for param in resnet18.parameters():
    param.requires_grad = False


for param in resnet18.layer4.parameters():
    param.requires_grad = True

for param in resnet18.fc.parameters():
    param.requires_grad = True


print("layer4 and fc are trainable.")


layer4 and fc are trainable.


In [32]:
optimizer = optim.Adam(
    filter(lambda p : p.requires_grad,resnet18.parameters()),
    lr = 0.00001
)
print("Fine-tuning optimizer is ready.")

Fine-tuning optimizer is ready.


#Resnet18 
LAYER 4


In [33]:
num_epochs = 10

for epoch in range(num_epochs):

    # =====================
    # Training
    # =====================
    resnet18.train()

    running_train_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = resnet18(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

    train_loss = running_train_loss / len(train_loader)


    # =====================
    # Validation
    # =====================
    resnet18.eval()

    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = resnet18(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}%"
    )

KeyboardInterrupt: 

In [34]:
for param in resnet18.parameters():
    param.requires_grad = False

for param in resnet18.layer3.parameters():
    param.requires_grad = True

for param in resnet18.layer4.parameters():
    param.requires_grad = True

for param in resnet18.fc.parameters():
    param.requires_grad = True


print("layer3 and layer4 and fc are trainable.")


layer3 and layer4 and fc are trainable.


In [35]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, resnet18.parameters()),
    lr=1e-5
)

print("Fine-tuning optimizer updated.")

Fine-tuning optimizer updated.


In [36]:
num_epochs = 10
best_val_accuracy = 0.0

for epoch in range(num_epochs):

    # Training
    resnet18.train()

    running_train_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = resnet18(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

    train_loss = running_train_loss / len(train_loader)


    # Validation
    resnet18.eval()

    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = resnet18(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}%"
    )


# save the best model

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save(
            resnet18.state_dict(),
            "resnet18_layer3_best.pth"
        )


        print(
            f"Best model saved! "
            f"Val Accuracy: {best_val_accuracy:.2f}%"
        )



KeyboardInterrupt: 

In [37]:
best_model = models.resnet18(weights=None)

best_model.fc = nn.Linear(
    best_model.fc.in_features,
    8
)

best_model.load_state_dict(
    torch.load("resnet18_layer3_best.pth")
)

best_model = best_model.to(device)
best_model.eval()

print("Best model loaded successfully.")


Best model loaded successfully.


In [38]:
correct = 0
total = 0

best_model.eval()

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = best_model(images)

        predictions = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

val_accuracy = 100 * correct / total

print(f"Validation Accuracy: {val_accuracy:.2f}%")

Validation Accuracy: 95.84%


In [39]:
all_predictions = []
all_labels = []

best_model.eval()

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = best_model(images)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Predictions:", len(all_predictions))
print("Labels:", len(all_labels))

Predictions: 385
Labels: 385


In [40]:
from sklearn.metrics import classification_report

class_names = [
    "ambulance",
    "autobus",
    "kamyun",
    "kamyunet",
    "minibus",
    "savari",
    "taxi",
    "vanet"
]

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4
    )
)

              precision    recall  f1-score   support

   ambulance     1.0000    0.9767    0.9882        43
     autobus     0.9615    1.0000    0.9804        50
      kamyun     0.9714    0.8500    0.9067        40
    kamyunet     0.9355    1.0000    0.9667        58
     minibus     0.9783    1.0000    0.9890        45
      savari     0.8750    0.9800    0.9245        50
        taxi     1.0000    0.9388    0.9684        49
       vanet     0.9783    0.9000    0.9375        50

    accuracy                         0.9584       385
   macro avg     0.9625    0.9557    0.9577       385
weighted avg     0.9607    0.9584    0.9582       385



In [ ]:
# from sklearn.metrics import confusion_matrix
# import matplotlib.pyplot as plt

# cm = confusion_matrix(all_labels,all_predictions)

# plt.figure(figsize=(8, 6))
# plt.imshow(cm)

# plt.xticks(range(len(class_names)), class_names, rotation=45)
# plt.yticks(range(len(class_names)), class_names)

# plt.xlabel("Predicted Label")
# plt.ylabel("True Label")
# plt.title("Confusion Matrix")

# plt.colorbar()

# plt.tight_layout()
# plt.show()

In [2]:
import torch

x = torch.randn(1, 3, 224, 224).to("cuda")

print("Tensor created:", x.shape)
print("Device:", x.device)

del x
torch.cuda.empty_cache()

print("GPU test OK")

Tensor created: torch.Size([1, 3, 224, 224])
Device: cuda:0
GPU test OK


In [3]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_model = models.resnet18(weights=None)

best_model.fc = nn.Linear(
    best_model.fc.in_features,
    8
)

best_model.load_state_dict(
    torch.load(
        "resnet18_layer3_best.pth",
        map_location=device
    )
)

best_model = best_model.to(device)
best_model.eval()

print("Best model loaded successfully.")
print("Device:", device)

Best model loaded successfully.
Device: cuda


In [4]:
import pickle
import sys
from pathlib import Path

with open("vehicle_split.pkl", "rb") as f:
    data = pickle.load(f)

X_train = data["X_train"]
X_val = data["X_val"]
y_train = data["y_train"]
y_val = data["y_val"]

sys.path.append(str(Path.cwd().parent))

from src.data_loader import create_dataloaders

train_loader, val_loader = create_dataloaders(
    X_train,
    X_val,
    y_train,
    y_val,
    batch_size=32
)

print("Validation batches:", len(val_loader))

Validation batches: 13


In [5]:
images, labels = next(iter(val_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

images = images.to(device)

with torch.no_grad():
    outputs = best_model(images)

print("Outputs:", outputs.shape)

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])
Outputs: torch.Size([32, 8])


In [6]:
all_predictions = []
all_labels = []

best_model.eval()

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        
        outputs = best_model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

print("Predictions:", len(all_predictions))
print("Labels:", len(all_labels))

Predictions: 385
Labels: 385


In [7]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_predictions)

print(cm)

[[42  0  0  0  0  0  0  1]
 [ 0 50  0  0  0  0  0  0]
 [ 0  1 34  4  1  0  0  0]
 [ 0  0  0 58  0  0  0  0]
 [ 0  0  0  0 45  0  0  0]
 [ 0  1  0  0  0 49  0  0]
 [ 0  0  0  0  0  3 46  0]
 [ 0  0  1  0  0  4  0 45]]


In [8]:
wrong_indices = [
    i for i, (true_label, pred_label)
    in enumerate(zip(all_labels, all_predictions))
    if true_label == 2 and pred_label == 3
]

print("Number of errors:", len(wrong_indices))

for i in wrong_indices:
    print(X_val[i])

Number of errors: 4
C:\Users\ASUS\Desktop\مکتب شریف\مکتب 145\پروژه دوم\data\train - merge - clean\kamyun\218963215.jpg
C:\Users\ASUS\Desktop\مکتب شریف\مکتب 145\پروژه دوم\data\train - merge - clean\kamyun\216045038.jpg
C:\Users\ASUS\Desktop\مکتب شریف\مکتب 145\پروژه دوم\data\train - merge - clean\kamyun\200170189.jpg
C:\Users\ASUS\Desktop\مکتب شریف\مکتب 145\پروژه دوم\data\train - merge - clean\kamyun\218884618.jpg


In [9]:
wrong_indices_vanet_savari = [
    i for i, (true_label, pred_label)
    in enumerate(zip(all_labels, all_predictions))
    if true_label == 7 and pred_label == 5
]

print("Number of errors:", len(wrong_indices_vanet_savari))

for i in wrong_indices_vanet_savari:
    print(X_val[i])

Number of errors: 4
C:\Users\ASUS\Desktop\مکتب شریف\مکتب 145\پروژه دوم\data\train - merge - clean\vanet\198100361.jpg
C:\Users\ASUS\Desktop\مکتب شریف\مکتب 145\پروژه دوم\data\train - merge - clean\vanet\214726285.jpg
C:\Users\ASUS\Desktop\مکتب شریف\مکتب 145\پروژه دوم\data\train - merge - clean\vanet\217178596.jpg
C:\Users\ASUS\Desktop\مکتب شریف\مکتب 145\پروژه دوم\data\train - merge - clean\vanet\216618067.jpg


# Resnet50

In [14]:
from torchvision import models



resnet50 = models.resnet50(weights= "DEFAULT")



print(resnet50)

resnet50.fc = nn.Linear(
resnet50.fc.in_features,
8
)

print(resnet50.fc)

for param in resnet50.parameters():
    param.requires_grad = False



for param in resnet50.fc.parameters():
    param.requires_grad = True


print("Backbone frozen.")
print("Final layer trainable.")

resnet50 = resnet50.to(device)
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    resnet50.fc.parameters(),
    lr = 0.001
)


print("ResNet50 is ready for training.")

images,labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = resnet50(images)

print("Images shape:", images.shape)
print("Outputs shape:", outputs.shape)
print("Labels shape:", labels.shape)



ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [15]:
num_epochs = 10

for epoch in range(num_epochs):

    # =====================
    # Training
    # =====================
    resnet50.train()

    running_train_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = resnet50(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

    train_loss = running_train_loss / len(train_loader)


    # =====================
    # Validation
    # =====================
    resnet50.eval()

    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = resnet50(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = 100 * correct / total


    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}%"
    )

Epoch [1/10] Train Loss: 1.6453 | Val Loss: 1.3271 | Val Accuracy: 75.58%
Epoch [2/10] Train Loss: 1.0363 | Val Loss: 0.9598 | Val Accuracy: 81.04%
Epoch [3/10] Train Loss: 0.7904 | Val Loss: 0.7777 | Val Accuracy: 85.19%
Epoch [4/10] Train Loss: 0.6349 | Val Loss: 0.6877 | Val Accuracy: 86.49%
Epoch [5/10] Train Loss: 0.5595 | Val Loss: 0.6140 | Val Accuracy: 86.49%
Epoch [6/10] Train Loss: 0.4769 | Val Loss: 0.5764 | Val Accuracy: 88.57%
Epoch [7/10] Train Loss: 0.4219 | Val Loss: 0.6503 | Val Accuracy: 88.83%
Epoch [8/10] Train Loss: 0.3946 | Val Loss: 0.5169 | Val Accuracy: 89.87%
Epoch [9/10] Train Loss: 0.3422 | Val Loss: 0.5253 | Val Accuracy: 89.35%
Epoch [10/10] Train Loss: 0.3281 | Val Loss: 0.4679 | Val Accuracy: 88.83%


FINETUNING LAYER 4

In [17]:
for param in resnet50.parameters():
    param.requires_grad = False

for param in resnet50.layer4.parameters():
    param.requires_grad = True


for param in resnet50.fc.parameters():
    param.requires_grad = True

print("layer4 and fc are trainable.")

optimizer = optim.Adam(
    filter(lambda p : p.requires_grad,resnet50.parameters()),
    lr = 0.00001
)

print("Fine-tuning optimizer is ready.")



layer4 and fc are trainable.
Fine-tuning optimizer is ready.


In [18]:
num_epochs = 10

for epoch in range(num_epochs):

    # =====================
    # Training
    # =====================
    resnet50.train()

    running_train_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = resnet50(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

    train_loss = running_train_loss / len(train_loader)


    # =====================
    # Validation
    # =====================
    resnet50.eval()

    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = resnet50(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}%"
    )



Epoch [1/10] Train Loss: 0.2560 | Val Loss: 0.4110 | Val Accuracy: 89.87%
Epoch [2/10] Train Loss: 0.1979 | Val Loss: 0.3845 | Val Accuracy: 90.13%
Epoch [3/10] Train Loss: 0.1672 | Val Loss: 0.3975 | Val Accuracy: 90.13%
Epoch [4/10] Train Loss: 0.1442 | Val Loss: 0.3647 | Val Accuracy: 90.39%
Epoch [5/10] Train Loss: 0.1226 | Val Loss: 0.3600 | Val Accuracy: 89.61%
Epoch [6/10] Train Loss: 0.1046 | Val Loss: 0.3163 | Val Accuracy: 92.21%
Epoch [7/10] Train Loss: 0.0765 | Val Loss: 0.3597 | Val Accuracy: 90.13%
Epoch [8/10] Train Loss: 0.0784 | Val Loss: 0.2930 | Val Accuracy: 91.69%
Epoch [9/10] Train Loss: 0.0921 | Val Loss: 0.3024 | Val Accuracy: 90.39%
Epoch [10/10] Train Loss: 0.0629 | Val Loss: 0.3068 | Val Accuracy: 91.17%


Resnet 34